# 09 — Inference Exploration

Explores model generation quality — prompt sensitivity, temperature effects, generation speed, and qualitative comparisons across checkpoints.

Run after `make dpo SIZE=125m`.

In [ ]:
import sys
sys.path.insert(0, '..')

import time
import json
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import torch

RESULTS_DIR = Path('../results')
DATA_DIR = Path('../data')

## 1. Load model and tokenizer

In [ ]:
from transformers import AutoConfig, AutoModelForCausalLM, PreTrainedTokenizerFast
from model import SLMConfig, SLMForCausalLM
from tokenizers import Tokenizer

AutoConfig.register('slm', SLMConfig)
AutoModelForCausalLM.register(SLMConfig, SLMForCausalLM)

MODEL_DIR = RESULTS_DIR / 'slm-125m-dpo' / 'final'
TOKENIZER_PATH = DATA_DIR / 'tokenizer' / 'slm_tokenizer.json'

if MODEL_DIR.exists():
    model = SLMForCausalLM.from_pretrained(
        str(MODEL_DIR),
        torch_dtype=torch.bfloat16,
    )
    model.eval()
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = model.to(device)
    tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Model loaded — {n_params/1e6:.1f}M params on {device}")
else:
    print(f"Model not found at {MODEL_DIR}")
    print("Run: make dpo SIZE=125m")
    model = None
    tokenizer = None

In [ ]:
def generate(prompt, max_new_tokens=200, temperature=0.7, top_p=0.9,
             do_sample=True, repetition_penalty=1.1):
    if model is None:
        return "Model not loaded"
    input_ids = torch.tensor([tokenizer.encode(prompt).ids]).to(device)
    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=do_sample,
            repetition_penalty=repetition_penalty,
            pad_token_id=0,
            eos_token_id=3,
        )
    new_tokens = output[0][input_ids.shape[1]:]
    return tokenizer.decode(new_tokens.tolist(), skip_special_tokens=True).strip()

def chat_prompt(user, system="You are a helpful assistant."):
    return f"<|system|>{system}<|endofturn|><|user|>{user}<|endofturn|><|assistant|>"

print("Helper functions defined")

## 2. Qualitative generation examples

In [ ]:
if model:
    examples = [
        ("General knowledge",  "What causes the northern lights?"),
        ("Reasoning",          "If a train travels 60mph for 2.5 hours, how far does it go? Show your work."),
        ("Coding",             "Write a Python function that checks if a string is a palindrome."),
        ("Creative",           "Write a haiku about training a language model."),
    ]

    for label, question in examples:
        prompt = chat_prompt(question)
        response = generate(prompt, max_new_tokens=200)
        print(f"{'='*60}")
        print(f"[{label.upper()}]")
        print(f"Q: {question}")
        print(f"A: {response}")
        print()

## 3. Temperature sensitivity

In [ ]:
if model:
    prompt = chat_prompt("Explain what attention is in transformers in one paragraph.")
    temperatures = [0.1, 0.5, 0.7, 1.0, 1.5]

    print("Temperature sensitivity (same prompt, different temperatures)")
    print()
    for temp in temperatures:
        response = generate(prompt, max_new_tokens=100, temperature=temp)
        print(f"T={temp}: {response[:200]}{'...' if len(response) > 200 else ''}")
        print()

## 4. Generation speed benchmark

In [ ]:
if model:
    prompt = chat_prompt("Tell me about the history of computing.")
    input_ids = torch.tensor([tokenizer.encode(prompt).ids]).to(device)
    token_counts = [50, 100, 200, 400]
    times = []
    tps_list = []

    print(f"{'Max tokens':>12}  {'Time (s)':>10}  {'Tokens/sec':>12}")
    print("-" * 38)

    for n_tokens in token_counts:
        torch.cuda.synchronize() if device == 'cuda' else None
        start = time.time()
        with torch.no_grad():
            output = model.generate(
                input_ids,
                max_new_tokens=n_tokens,
                do_sample=False,
                pad_token_id=0,
                eos_token_id=3,
            )
        torch.cuda.synchronize() if device == 'cuda' else None
        elapsed = time.time() - start
        actual_new = output.shape[1] - input_ids.shape[1]
        tps = actual_new / elapsed
        times.append(elapsed)
        tps_list.append(tps)
        print(f"  {n_tokens:>10}  {elapsed:>10.2f}  {tps:>12.1f}")

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(token_counts, times, 'o-', color='#4C72B0', linewidth=2)
    axes[0].set_xlabel('Max new tokens')
    axes[0].set_ylabel('Time (s)')
    axes[0].set_title('Generation Time', fontsize=11, fontweight='bold')

    axes[1].plot(token_counts, tps_list, 'o-', color='#55A868', linewidth=2)
    axes[1].set_xlabel('Max new tokens')
    axes[1].set_ylabel('Tokens/sec')
    axes[1].set_title('Throughput', fontsize=11, fontweight='bold')

    fig.suptitle(f'Generation Speed — slm-125m on {device}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 5. Multi-turn conversation

In [ ]:
if model:
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
    ]

    conversation = [
        "What is gradient descent?",
        "Can you give me a simple code example?",
        "What is the learning rate and why does it matter?",
    ]

    def format_messages(msgs):
        parts = []
        for m in msgs:
            if m['role'] == 'system':
                parts.append(f"<|system|>{m['content']}<|endofturn|>")
            elif m['role'] == 'user':
                parts.append(f"<|user|>{m['content']}<|endofturn|>")
            elif m['role'] == 'assistant':
                parts.append(f"<|assistant|>{m['content']}<|endofturn|>")
        parts.append("<|assistant|>")
        return "".join(parts)

    for user_turn in conversation:
        messages.append({"role": "user", "content": user_turn})
        prompt = format_messages(messages)
        response = generate(prompt, max_new_tokens=150)
        messages.append({"role": "assistant", "content": response})

        print(f"User:      {user_turn}")
        print(f"Assistant: {response[:300]}{'...' if len(response) > 300 else ''}")
        print()

## 6. Greedy vs sampled output comparison

In [ ]:
if model:
    prompt = chat_prompt("What are the key components of a transformer architecture?")

    print("GREEDY (deterministic):")
    print(generate(prompt, max_new_tokens=150, do_sample=False))

    print()
    print("SAMPLED (T=0.7, top_p=0.9) — run 1:")
    print(generate(prompt, max_new_tokens=150, temperature=0.7))

    print()
    print("SAMPLED (T=0.7, top_p=0.9) — run 2:")
    print(generate(prompt, max_new_tokens=150, temperature=0.7))

## 7. Token probability inspection

In [ ]:
if model:
    import torch.nn.functional as F

    prompt = chat_prompt("The capital of France is")
    input_ids = torch.tensor([tokenizer.encode(prompt).ids]).to(device)

    with torch.no_grad():
        output = model(input_ids)

    # Get probabilities for the next token
    next_token_logits = output.logits[0, -1, :]
    probs = F.softmax(next_token_logits, dim=-1)
    top_probs, top_ids = torch.topk(probs, 15)

    print(f"Prompt: '{prompt[-50:]}'")
    print(f"\nTop 15 next token predictions:")
    print(f"  {'Token':<20}  {'Probability':>12}")
    print("-" * 36)
    for prob, tid in zip(top_probs.tolist(), top_ids.tolist()):
        token = tokenizer.id_to_token(tid)
        print(f"  {repr(token):<20}  {prob:>12.4f}")